# Phase 8: Interpretable Thermal-Risk Layer

This notebook loads persisted Phase 8 risk artifacts over the frozen Ridge/OLS forecasts. It does not retrain models or alter Phases 0-7.

## Frozen contract and central quantity

The selected models remain Extended 21-feature Ridge(alpha=1.0) at 60s and Extended 21-feature OLS at 300s. The central state is predicted_future_temperature = current_T_cell_max + predicted_DeltaT. The 70 C cutoff is documented in the project assessment; intermediate bands are validation-derived descriptive thresholds, not certified safety limits.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
results_dir = project_root / 'results' / 'freetwinev'
figures_dir = results_dir / 'figures'
thresholds = json.loads((results_dir / 'phase8_thresholds.json').read_text(encoding='utf-8'))
validation = pd.read_csv(results_dir / 'phase8_risk_predictions_validation.csv')
test = pd.read_csv(results_dir / 'phase8_risk_predictions_test.csv')
validation_metrics = pd.read_csv(results_dir / 'phase8_risk_metrics_validation.csv')
test_metrics = pd.read_csv(results_dir / 'phase8_risk_metrics_test.csv')
audit = json.loads((results_dir / 'phase8_audit.json').read_text(encoding='utf-8'))
print('Audit:', audit['overall_status'])

## Validation-only threshold derivation

P75/P90/P99 temperature bands and P90 rise/trend/spread thresholds were derived from ID16 and ID22 only. A 12-sample trailing persistence window equals 60 seconds. The 70 C value is the documented engineering cutoff; it was not learned from test data.

In [ ]:
thresholds

## Risk logic

Normal/Elevated/High temperature bands use validation P75/P90. Critical is predicted future temperature at or above 70 C. Transparent secondary indicators cover rapid predicted rise, recent 30s trend, wide cell spread, and sustained elevated temperature. Secondary indicators can escalate a descriptive category but are not combined into an opaque score.

In [ ]:
validation_metrics

In [ ]:
validation[['split', 'exp_id', 'time_s', 'horizon', 'current_T_cell_max', 'target_predicted', 'predicted_future_temperature', 'T_cell_max_diff_30s', 'T_cell_spread', 'risk_category']].head()

## Held-out test reporting

ID18 and ID19 are shown separately only after validation thresholds and logic were frozen. These rows are held-out evaluation and were not used to tune categories, select features, or calibrate the model.

In [ ]:
test_metrics

In [ ]:
test[['split', 'exp_id', 'time_s', 'horizon', 'current_T_cell_max', 'target_predicted', 'predicted_future_temperature', 'risk_category', 'secondary_signal_count', 'sustained_elevated_60s']].head()

## Risk timelines and distributions

The runner-generated figures show predicted future temperature, category counts, validation/test timelines, and non-normal risk duration.

In [ ]:
names = ['41_phase8_temperature_risk_overview.png', '42_phase8_validation_risk_timeline.png', '43_phase8_risk_duration.png', '44_phase8_test_risk_timeline_posthoc.png']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, name in zip(axes.flat, names):
    ax.imshow(plt.imread(figures_dir / name)); ax.set_title(name); ax.axis('off')
plt.tight_layout()

## Audit and limitations

The audit verifies unchanged Phase 3-7 artifacts, exact run membership, validation-only thresholds, no refitting, no test tuning, temporal ordering, reconstruction consistency, and required outputs. Categories are interpretable descriptors, not certified safety decisions.

In [ ]:
assert audit['overall_status'] == 'PASS'
assert audit['threshold_source'] == 'validation runs ID16 and ID22 only'
assert audit['checks']['thresholds_validation_only']
assert audit['checks']['no_test_threshold_tuning']
assert audit['checks']['no_model_refitting']
audit['checks']

## Conclusion

Phase 8 converts frozen predicted thermal state into validation-derived descriptive risk categories and explicit trend/spread/persistence indicators. It does not claim certified safety, alter forecasting models, or begin Phase 9 cooling recommendation.